# ADX < 15 Mean Reversion On on SPY
## Strategy Brief
This strategy aims to capitalize on mean reversion in the SPY ETF when the Average Directional Index (ADX) is below 15, indicating a non-trending market. The signal is generated when the ADX falls below 15, predicting that the market will revert to its mean. The trade logic involves entering a long position when the signal is triggered and exiting when the ADX rises above 15. Historically, this strategy has shown potential for profitability during low-volatility periods.
## References
- https://www.adx.faa.gov/portal/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters and constants required for the strategy. These include the lookback period for the ADX calculation and the threshold value for the ADX signal.

In [ ]:
ADX_LOOKBACK = 14
ADX_THRESHOLD = 15
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
TICKER = 'SPY'

## PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance, calculate the ADX indicator, and plot it alongside the price to visualize potential signals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Calculate ADX
high = data['High']
low = data['Low']
close = data['Close']

plus_dm = high.diff()
minus_dm = low.diff()

plus_dm[plus_dm < 0] = 0
minus_dm[minus_dm > 0] = 0

tr = np.maximum(high - low, np.maximum(abs(high - close.shift()), abs(low - close.shift())))
atr = tr.rolling(window=ADX_LOOKBACK).mean()

plus_di = 100 * (plus_dm.ewm(alpha=1/ADX_LOOKBACK).mean() / atr)
minus_di = abs(100 * (minus_dm.ewm(alpha=1/ADX_LOOKBACK).mean() / atr))

dx = (abs(plus_di - minus_di) / (plus_di + minus_di)) * 100
adx = dx.rolling(window=ADX_LOOKBACK).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data.index, data['Close'], label='SPY Close')
plt.plot(data.index, adx, label='ADX', linestyle='--')
plt.axhline(ADX_THRESHOLD, color='red', linestyle='--', label='ADX Threshold')
plt.title('SPY Price and ADX')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
In this phase, we create the signal based on the ADX indicator. We generate a signal to go long when the ADX is below the threshold and exit when it rises above.

In [ ]:
# Generate signals
signal = (adx < ADX_THRESHOLD).astype(int)

# Entry/Exit logic
positions = signal.diff().fillna(0)

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating the daily returns and plotting the equity curve based on the generated signals.

In [ ]:
# Calculate daily returns
daily_returns = data['Close'].pct_change().fillna(0)

# Calculate strategy returns
strategy_returns = daily_returns * positions.shift(1)

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Strategy Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. A comparison is made against a buy-and-hold strategy.

In [ ]:
def calculate_performance(equity_curve):
    # CAGR
    years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0]) ** (1/years) - 1
    
    # Sharpe Ratio
    sharpe_ratio = (strategy_returns.mean() / strategy_returns.std()) * np.sqrt(252)
    
    # Sortino Ratio
    downside_returns = strategy_returns[strategy_returns < 0]
    sortino_ratio = (strategy_returns.mean() / downside_returns.std()) * np.sqrt(252)
    
    # Calmar Ratio
    max_drawdown = (equity_curve / equity_curve.cummax()).min() - 1
    calmar_ratio = cagr / abs(max_drawdown)
    
    # Buy and Hold
    buy_and_hold_returns = (1 + daily_returns).cumprod()
    buy_and_hold_cagr = (buy_and_hold_returns.iloc[-1] / buy_and_hold_returns.iloc[0]) ** (1/years) - 1
    
    return {
        'CAGR': cagr,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Calmar Ratio': calmar_ratio,
        'Max Drawdown': max_drawdown,
        'Buy and Hold CAGR': buy_and_hold_cagr
    }

performance = calculate_performance(equity_curve)
performance_df = pd.DataFrame(performance, index=[0])
print(performance_df)

## PHASE 6 - Deploy & Monitor
We create a function to download the latest 60 days of SPY data, compute today's signal, and print the position for potential deployment.

In [ ]:
def get_latest_signal():
    latest_data = yf.download(TICKER, period='60d')
    high = latest_data['High']
    low = latest_data['Low']
    close = latest_data['Close']
    
    plus_dm = high.diff()
    minus_dm = low.diff()
    plus_dm[plus_dm < 0] = 0
    minus_dm[minus_dm > 0] = 0
    
    tr = np.maximum(high - low, np.maximum(abs(high - close.shift()), abs(low - close.shift())))
    atr = tr.rolling(window=ADX_LOOKBACK).mean()
    plus_di = 100 * (plus_dm.ewm(alpha=1/ADX_LOOKBACK).mean() / atr)
    minus_di = abs(100 * (minus_dm.ewm(alpha=1/ADX_LOOKBACK).mean() / atr))
    dx = (abs(plus_di - minus_di) / (plus_di + minus_di)) * 100
    adx = dx.rolling(window=ADX_LOOKBACK).mean()
    
    signal = (adx < ADX_THRESHOLD).astype(int)
    current_signal = signal.iloc[-1]
    
    print(f"Current Signal: {'Long' if current_signal == 1 else 'Cash'}")

get_latest_signal()